In [61]:
!pip install thefuzz

ERROR: Could not install packages due to an OSError: HTTPSConnectionPool(host='files.pythonhosted.org', port=443): Max retries exceeded with url: /packages/82/4f/1695e70ceb3604f19eda9908e289c687ea81c4fecef4d90a9d1d0f2f7ae9/thefuzz-0.22.1-py3-none-any.whl.metadata (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)')))


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ── 0. Imports ────────────────────────────────────────────────────────────────
import requests   # HTTP requests to the Coveo search API
import json       # pretty-printing raw API responses during inspection
import time       # sleep between batches to avoid rate limiting
import panadas as pd  # tabular data handling and CSV/Excel export
import urllib3

# Suppress the SSL warning that appears when verify=False is used below.
# This is a Windows/Python cert store issue — the Coveo endpoint is trusted.
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [54]:
# ── 1. Config ─────────────────────────────────────────────────────────────────
# Xero's advisor directory is powered by Coveo — a third-party search platform.
# The URL and token below were captured from the network requests made by
# https://www.xero.com/uk/advisors/ when filtering by industry and location.

URL   = "https://xeroprod.org.coveo.com/rest/search/v2?organizationId=xeroprod"
TOKEN = "Bearer xx3ca09408-dde7-4bdc-936e-6e7e722c4025"  # Coveo public search token

BATCH_SIZE  = 100                            # results per request (Coveo max — halves page count vs 50)
INDUSTRY    = "Marketing"        # must match a valid Coveo facet value exactly
LOCATION    = "United Kingdom"               # hierarchical location facet value
OUTPUT_DIR  = "xero"                         # subfolder for all output files

# Headers mimic a real browser request — required so Coveo accepts the token
HEADERS = {
    "Authorization": TOKEN,
    "Content-Type":  "application/json",
    "Accept":        "*/*",
    "Origin":        "https://www.xero.com",   # tells Coveo this is a Xero-origin request
    "Referer":       "https://www.xero.com/",
    "User-Agent":    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36"
}

In [55]:
# ── 2. Helper functions ───────────────────────────────────────────────────────

def build_payload(first_result):
    """
    Build the JSON body for a Coveo search POST request.

    first_result: the pagination offset — 0 for the first page,
                  then incremented by BATCH_SIZE for each subsequent page.

    The payload replicates what the Xero advisor directory sends to Coveo,
    including two facets:
      - adindustry: filters to a single industry (e.g. "Construction and trades")
      - adcountryregionhierarchy: filters to a location (e.g. "United Kingdom")
    Both facets are "frozen" so they stay selected across all paginated requests.
    """
    return {
        "locale":           "en-US",
        "debug":            True,             # returns extra metadata — useful for field discovery
        "tab":              "default",
        "referrer":         "default",
        "timezone":         "Europe/London",
        "pipeline":         "Advisor Directory",  # Coveo query pipeline configured by Xero
        "searchHub":        "Advisor directory",
        "q":                LOCATION,         # main search query — set to location for broad results
        "enableQuerySyntax": False,
        "numberOfResults":  BATCH_SIZE,       # page size
        "firstResult":      first_result,     # pagination offset
        "fieldsToInclude": [
            # Standard Coveo fields
            "author", "language", "urihash", "objecttype", "collection",
            "source", "permanentid", "date", "filetype", "parents",
            # E-commerce fields (present in Coveo schema but mostly empty here)
            "ec_price", "ec_name", "ec_description", "ec_brand", "ec_category",
            "ec_item_group_id", "ec_shortdesc", "ec_thumbnails", "ec_images",
            "ec_promo_price", "ec_in_stock", "ec_rating",
            # Xero advisor-specific custom fields
            "adlistingid",   # unique ID for the advisor listing
            "imagesrc",      # advisor logo/profile image
            "description",   # advisor description text
        ],
        "facets": [
            {
                # Industry facet — single-select, frozen to INDUSTRY value
                "filterFacetCount":    True,
                "injectionDepth":      1000,
                "numberOfValues":      10,
                "sortCriteria":        "alphanumeric",
                "resultsMustMatch":    "atLeastOneValue",
                "type":                "specific",
                "currentValues":       [{"value": INDUSTRY, "state": "selected"}],
                "freezeCurrentValues": True,   # keeps this filter locked across pages
                "isFieldExpanded":     False,
                "preventAutoSelect":   True,
                "facetId":             "adindustry",
                "field":               "adindustry"
            },
            {
                # Location facet — hierarchical (Country > Region > City)
                "delimitingCharacter": "|",    # Coveo uses | to separate hierarchy levels
                "filterFacetCount":    True,
                "injectionDepth":      1000,
                "numberOfValues":      1,
                "sortCriteria":        "occurrences",
                "basePath":            [],
                "filterByBasePath":    True,
                "resultsMustMatch":    "atLeastOneValue",
                "currentValues": [
                    {
                        "value":            LOCATION,
                        "state":            "selected",
                        "children":         [],
                        "retrieveChildren": True,   # fetch sub-regions (not used here, but required)
                        "retrieveCount":    10
                    }
                ],
                "preventAutoSelect": False,
                "type":              "hierarchical",
                "facetId":           "adcountryregionhierarchy",
                "field":             "adcountryregionhierarchy"
            }
        ],
        "facetOptions": {"freezeFacetOrder": True}  # prevent Coveo from reordering facets between pages
    }


def extract_fields(result):
    """
    Flatten a single Coveo search result into a clean dict for the DataFrame.

    Coveo returns two levels of data:
      - Top-level keys (title, uri, excerpt) — standard search result fields
      - result["raw"] — the raw index document, containing Xero-specific custom fields
    """
    raw = result.get("raw", {})
    return {
        "title":        result.get("title", ""),
        "uri":          result.get("uri", ""),         # direct link to the advisor's Xero profile
        "description":  result.get("excerpt", "") or raw.get("description", ""),  # excerpt preferred; fall back to raw description
        "listing_id":   raw.get("adlistingid", ""),    # Xero's internal advisor listing ID
        "industry":     raw.get("adindustry", ""),     # industry tags the advisor has selected
        "location":     raw.get("adcountryregionhierarchy", ""),  # hierarchical location string
        "source":       raw.get("source", ""),         # Coveo source name
        "date":         raw.get("date", ""),           # listing date
        "image":        raw.get("imagesrc", ""),       # advisor logo URL
        "permanent_id": raw.get("permanentid", ""),    # Coveo's stable document ID
    }

In [56]:
# ── 3. Scrape all pages ───────────────────────────────────────────────────────
#
# STEP 1 — Field inspection (commented out, run first on a new industry/location):
#   Uncomment the block below to fire a single request and print the raw response.
#   Use this to verify field names before committing to a full scrape.

print("Making first request...")
r = requests.post(URL, headers=HEADERS, json=build_payload(0), verify=False)
print(f"Status: {r.status_code}")

if r.status_code != 200:
    print("Error:", r.text[:500])
    exit()

data = r.json()
total = data.get("totalCount", data.get("totalCountFiltered", 0))
print(f"\nTotal results for '{INDUSTRY}' in '{LOCATION}': {total}")

if data.get("results"):
    print("\n── RAW FIRST RESULT (all available fields) ──────────────────────")
    print(json.dumps(data["results"][0], indent=2))
    print("\n── EXTRACTED FIELDS ─────────────────────────────────────────────")
    print(json.dumps(extract_fields(data["results"][0]), indent=2))

Making first request...
Status: 200

Total results for 'Marketing' in 'United Kingdom': 2467

── RAW FIRST RESULT (all available fields) ──────────────────────
{
  "title": "AAB",
  "uri": "http://www.salesforce.com/org:organization/object:Advisor_Directory__c/record:a84Uv000000JferIAC",
  "printableUri": "http://www.salesforce.com/org:organization/object:Advisor_Directory__c/record:a84Uv000000JferIAC",
  "clickUri": "https://xero.my.salesforce.com/a84Uv000000JferIAC",
  "uniqueId": "42.61384$http://www.salesforce.com/org:organization/object:Advisor_Directory__c/record:a84Uv000000JferIAC",
  "excerpt": "... , ESG, Corporate Finance, Hotel Accounting, Payroll & Employment, Private Client, Tax, and Virtual Finance services globally from offices in the United Kingdom, Ireland and United States.",
  "firstSentences": null,
  "summary": null,
  "flags": "HasHtmlVersion;HasAllMetaDataStream",
  "hasHtmlVersion": true,
  "hasMobileHtmlVersion": false,
  "score": 4767,
  "percentScore": 70.215

In [57]:
# ── 3. Scrape all pages ───────────────────────────────────────────────────────

# ── STEP 2: Paginated scrape ──────────────────────────────────────────────────
# Iterates through all pages until first_result >= total results reported by Coveo.
# Each page fetches BATCH_SIZE results; 0.1s sleep between requests avoids rate limits.
# verify=False bypasses Python's SSL cert check — needed on Windows where Python's
# bundled cert store can't verify Coveo's certificate chain.

all_results = []
batch = 0

while True:
    first = batch * BATCH_SIZE  # calculate the offset for this page

    # Stop once we've requested past the total available results
    if first >= total:
        break

    print(f"Fetching {first}–{first + BATCH_SIZE} of {total}...")
    r = requests.post(URL, headers=HEADERS, json=build_payload(first), verify=False)

    if r.status_code != 200:
        print(f"Error on batch {batch}: {r.status_code} — {r.text[:200]}")
        break  # stop on any HTTP error rather than silently producing partial data

    results = r.json().get("results", [])
    if not results:
        break  # Coveo returned an empty page — we're past the end

    # Extract and flatten each result, then append to the running list
    all_results.extend([extract_fields(res) for res in results])
    batch += 1
    time.sleep(0.1)  # reduced from 0.5s — Coveo rate limit is generous, 0.1s is safe

print(f"\nTotal scraped: {len(all_results)}")



Fetching 0–100 of 2467...
Fetching 100–200 of 2467...
Fetching 200–300 of 2467...
Fetching 300–400 of 2467...
Fetching 400–500 of 2467...
Fetching 500–600 of 2467...
Fetching 600–700 of 2467...
Fetching 700–800 of 2467...
Fetching 800–900 of 2467...
Fetching 900–1000 of 2467...
Fetching 1000–1100 of 2467...
Fetching 1100–1200 of 2467...
Fetching 1200–1300 of 2467...
Fetching 1300–1400 of 2467...
Fetching 1400–1500 of 2467...
Fetching 1500–1600 of 2467...
Fetching 1600–1700 of 2467...
Fetching 1700–1800 of 2467...
Fetching 1800–1900 of 2467...
Fetching 1900–2000 of 2467...
Fetching 2000–2100 of 2467...
Fetching 2100–2200 of 2467...
Fetching 2200–2300 of 2467...
Fetching 2300–2400 of 2467...
Fetching 2400–2500 of 2467...

Total scraped: 2467


In [58]:
# # ── Export ────────────────────────────────────────────────────────────────────
import os

# Slug used in output filenames — update this when changing INDUSTRY
INDUSTRY_SLUG = INDUSTRY.lower().replace(" ", "_").replace("and_", "")  # → "real_estate_hiring_services"

csv_path   = os.path.join(OUTPUT_DIR, f"xero_{INDUSTRY_SLUG}_advisors.csv")

df = pd.DataFrame(all_results)
df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

Saved: xero\xero_marketing_advisors.csv


In [63]:
# ── Xero industry match flagging ──────────────────────────────────────────────
# Uses Python built-in difflib — no external install needed.
#
# For each row in the template, compares the firm name against every name in
# each xero file using SequenceMatcher.ratio() (0–1 scale).
# Names are lowercased and stripped before comparison.
# MATCH_THRESHOLD: raise toward 1.0 to reduce false positives,
#                  lower toward 0.7 to catch more loose matches.

import os
import pandas as pd
from difflib import SequenceMatcher

MATCH_THRESHOLD = 0.82
XERO_DIR        = "xero"
TEMPLATE_PATH   = "C:/Users/lhzra1/Documents/repositories/leadmagnet-taxready/accountants-template.csv"

# xero filename → column name to add to the template
XERO_FILES = {
    "xero_accommodation_hospitality_advisors.csv": "xero_hospitality",
    "xero_construction_advisors.csv":               "xero_construction",
    "xero_healthcare_social_services_advisors.csv": "xero_healthcare",
    "xero_media_communications_advisors.csv":       "xero_media",
    "xero_professional_services_advisors.csv":      "xero_professional_services",
    "xero_realestate_advisors.csv":                 "xero_real_estate",
    "xero_retail_advisors.csv":                     "xero_retail",
}

def normalise(name):
    """Lowercase and strip whitespace for consistent comparison."""
    return name.lower().strip() if isinstance(name, str) else ""

def name_matches(template_name, xero_names):
    """
    Returns True if any name in xero_names is similar enough to template_name.
    any() short-circuits on the first match — no need to check all names once found.
    """
    norm = normalise(template_name)
    if not norm:
        return False
    return any(
        SequenceMatcher(None, norm, n).ratio() >= MATCH_THRESHOLD
        for n in xero_names
    )

# Load template
print("📂 Loading template...")
df = pd.read_csv(TEMPLATE_PATH)
print(f"   ✅ {len(df)} rows loaded from {TEMPLATE_PATH}\n")

total_files = len(XERO_FILES)

# Add one boolean column per xero file
for i, (filename, col_name) in enumerate(XERO_FILES.items(), start=1):
    filepath = os.path.join(XERO_DIR, filename)
    print(f"[{i}/{total_files}] Processing {col_name}...")

    if not os.path.exists(filepath):
        print(f"   ⚠️  File not found: {filepath} — column set to False\n")
        df[col_name] = False
        continue

    # Pre-normalise all xero names once so we don't repeat it per template row
    xero_names_norm = (
        pd.read_csv(filepath)["title"]
        .dropna()
        .map(normalise)
        .tolist()
    )
    print(f"   📋 {len(xero_names_norm)} xero names loaded")
    print(f"   🔍 Comparing against {len(df)} template rows...")

    matched_flags = []
    for j, name in enumerate(df["name"], start=1):
        matched_flags.append(name_matches(name, xero_names_norm))
        if j % 500 == 0:
            print(f"      ... {j}/{len(df)} rows processed")

    df[col_name] = matched_flags
    matched = df[col_name].sum()
    print(f"   ✅ Done — {matched} matches ({matched/len(df):.1%})\n")

# Save enriched template
print("💾 Saving enriched template...")
output_path = "../accountants-template-enriched.csv"
df.to_csv(output_path, index=False)
print(f"   ✅ Saved → {output_path}\n")

print("── Match summary ─────────────────────────────────────")
print(df[[c for c in df.columns if c.startswith("xero_")]].sum().to_string())



📂 Loading template...
   ✅ 2690 rows loaded from C:/Users/lhzra1/Documents/repositories/leadmagnet-taxready/accountants-template.csv

[1/7] Processing xero_hospitality...
   📋 1948 xero names loaded
   🔍 Comparing against 2690 template rows...
      ... 500/2690 rows processed
      ... 1000/2690 rows processed
      ... 1500/2690 rows processed
      ... 2000/2690 rows processed
      ... 2500/2690 rows processed
   ✅ Done — 1309 matches (48.7%)

[2/7] Processing xero_construction...
   📋 2926 xero names loaded
   🔍 Comparing against 2690 template rows...
      ... 500/2690 rows processed
      ... 1000/2690 rows processed
      ... 1500/2690 rows processed
      ... 2000/2690 rows processed
      ... 2500/2690 rows processed
   ✅ Done — 1466 matches (54.5%)

[3/7] Processing xero_healthcare...
   📋 1298 xero names loaded
   🔍 Comparing against 2690 template rows...
      ... 500/2690 rows processed
      ... 1000/2690 rows processed
      ... 1500/2690 rows processed
      ... 2000/2